# 42ndMind v1.9-safe4 → Qwen 0.5B LoRA Training

Clean Colab notebook for the second adapter test.

Use **Runtime → Change runtime type → T4 GPU** before running.

This notebook trains a small LoRA adapter on the `42ndMind v1_9-safe4` chat SFT dataset, then compares base-model output against adapter output on hidden contradiction prompts.

In [ ]:
# 0. Check GPU
!nvidia-smi

## 1. Install libraries

This removes `torchao` first because some Colab images ship an incompatible `torchao` version that can crash PEFT.

In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes

## 2. Clone 42ndAlignment

If this folder already exists because you reran the notebook, it will skip cloning.

In [ ]:
from pathlib import Path

repo = Path('/content/42ndAlignment')
if repo.exists():
    print('Repo already exists:', repo)
else:
    !git clone https://github.com/42ndMoose/42ndAlignment.git /content/42ndAlignment

## 3. Load or upload the v1_9 dataset

The notebook first checks whether the dataset already exists inside the cloned repo:

`datasets/42ndmind/v1_9/train.jsonl`

If it does not exist, upload `42ndmind_v1_9_safe4_dataset.zip` when the upload box appears.

In [ ]:
from pathlib import Path
import zipfile, shutil, os

repo = Path('/content/42ndAlignment')
data_dir = repo / 'datasets' / '42ndmind' / 'v1_9'
train_path = data_dir / 'train.jsonl'
eval_path = data_dir / 'eval.jsonl'

if train_path.exists() and eval_path.exists():
    print('Dataset already found in repo:')
    print(train_path)
    print(eval_path)
else:
    print('Dataset not found. Upload 42ndmind_v1_9_safe4_dataset.zip now.')
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(name for name in uploaded if name.endswith('.zip'))
    zip_path = Path('/content') / zip_name
    extract_dir = Path('/content/v1_9_dataset_extract')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_dir)

    found_train = list(extract_dir.rglob('train.jsonl'))
    found_eval = list(extract_dir.rglob('eval.jsonl'))
    if not found_train or not found_eval:
        raise FileNotFoundError('Uploaded zip does not contain train.jsonl and eval.jsonl')

    source_dir = found_train[0].parent
    data_dir.mkdir(parents=True, exist_ok=True)
    for p in source_dir.iterdir():
        if p.is_file():
            shutil.copy2(p, data_dir / p.name)
    print('Copied dataset files to:', data_dir)

print('Dataset files:')
for p in sorted(data_dir.iterdir()):
    print(' -', p.name)

## 4. Inspect dataset

Expected v1_9-safe4 size: about 152 train rows and 38 eval rows.

In [ ]:
import json
from pathlib import Path

train_path = Path('/content/42ndAlignment/datasets/42ndmind/v1_9/train.jsonl')
eval_path = Path('/content/42ndAlignment/datasets/42ndmind/v1_9/eval.jsonl')

train_rows = [json.loads(line) for line in train_path.read_text(encoding='utf-8').splitlines() if line.strip()]
eval_rows = [json.loads(line) for line in eval_path.read_text(encoding='utf-8').splitlines() if line.strip()]

print('Train rows:', len(train_rows))
print('Eval rows:', len(eval_rows))
print('Train keys:', train_rows[0].keys())
print('Roles in first row:', [m['role'] for m in train_rows[0]['messages']])
print('\nFirst row preview:')
print(json.dumps(train_rows[0], indent=2, ensure_ascii=False)[:3000])

## 5. Train the v1_9-safe4 LoRA adapter

This uses the same base model as the v1.8 smoke test so the comparison stays clean.

In [ ]:
import torch
from pathlib import Path
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
repo = Path('/content/42ndAlignment')
data_dir = repo / 'datasets' / '42ndmind' / 'v1_9'
output_dir = '/content/42ndAlignment/artifacts/42ndmind_v1_9_safe4_qwen05b_lora'

dataset = load_dataset('json', data_files={
    'train': str(data_dir / 'train.jsonl'),
    'eval': str(data_dir / 'eval.jsonl'),
})

print(dataset)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules='all-linear',
)

args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy='epoch',
    fp16=True,
    report_to='none',
    max_length=1024,
    packing=False,
)

trainer = SFTTrainer(
    model=model_id,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['eval'],
    peft_config=peft_config,
)

trainer.train()
trainer.save_model(output_dir)

print('Saved adapter to:', output_dir)

## 6. Compare base model vs adapter

The test prompts below are not exact training rows. The structured prompt should improve more than the plain-language prompt because the dataset is still mostly process-trace shaped.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
adapter_dir = '/content/42ndAlignment/artifacts/42ndmind_v1_9_safe4_qwen05b_lora'

tokenizer = AutoTokenizer.from_pretrained(model_id)

def generate_chat(model, messages, max_new_tokens=500):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = out[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='auto',
)
base_model.eval()

structured_test_messages = [
    {
        'role': 'system',
        'content': 'You are an epistemic runtime trained to reduce naive acceptance, preserve live hypotheses, detect contradiction, and update belief state from evidence.'
    },
    {
        'role': 'user',
        'content': '''{
  "prior_claim": {
    "text": "I returned the borrowed laptop yesterday.",
    "subject": "user",
    "action": "return",
    "object": "laptop",
    "time": "yesterday",
    "polarity": "positive",
    "scope": "bounded_or_unspecified",
    "confidence": 0.75,
    "status": "stated"
  },
  "new_claim": {
    "text": "The laptop is still in my room because I forgot to bring it back.",
    "subject": "user",
    "action": "possess",
    "object": "laptop",
    "time": "present",
    "polarity": "positive",
    "scope": "bounded_or_unspecified",
    "confidence": 0.85,
    "status": "stated"
  }
}'''
    }
]

plain_test_messages = [
    {
        'role': 'system',
        'content': 'You are an epistemic runtime trained to reduce naive acceptance, preserve live hypotheses, detect contradiction, and update belief state from evidence.'
    },
    {
        'role': 'user',
        'content': 'A user says he returned a borrowed laptop yesterday. Later he says the laptop is still in his room because he forgot to bring it back. Analyze the epistemic situation.'
    }
]

print('BASE MODEL STRUCTURED OUTPUT:\n')
print(generate_chat(base_model, structured_test_messages))

adapter_model = PeftModel.from_pretrained(base_model, adapter_dir)
adapter_model.eval()

print('\nADAPTER STRUCTURED OUTPUT:\n')
print(generate_chat(adapter_model, structured_test_messages))

print('\nBASE MODEL PLAIN OUTPUT:\n')
print(generate_chat(base_model, plain_test_messages))

print('\nADAPTER PLAIN OUTPUT:\n')
print(generate_chat(adapter_model, plain_test_messages))

## 7. Save/download adapter and report

The adapter zip is the artifact to keep in Drive/local storage. Avoid committing adapter weights to GitHub unless you use Git LFS or GitHub Releases.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

adapter_dir = Path('/content/42ndAlignment/artifacts/42ndmind_v1_9_safe4_qwen05b_lora')
zip_path = shutil.make_archive('/content/42ndmind_v1_9_safe4_qwen05b_lora', 'zip', adapter_dir)
print('Created:', zip_path)
files.download(zip_path)

## 8. Optional: create a short experiment report file

In [ ]:
from pathlib import Path
from google.colab import files

report_dir = Path('/content/42ndAlignment/docs/reports')
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / '42ndmind_v1_9_safe4_qwen05b_lora.md'

report = """# 42ndMind v1.9-safe4 Qwen 0.5B LoRA

Date: 2026-05-04

## Purpose

Train the second adapter using the larger v1.9-safe4 process-trace dataset exported from 42ndMind.

## Dataset

Path:

`datasets/42ndmind/v1_9/`

Files:

- `combined_alignment_sft.jsonl`
- `combined_preference_pairs.jsonl`
- `combined_manifest.json`
- `excluded_scenarios.json`
- `scenario_summary.json`
- `train.jsonl`
- `eval.jsonl`

Split:

- Train rows: 152
- Eval rows: 38

## Model

Base model:

`Qwen/Qwen2.5-0.5B-Instruct`

Method:

LoRA SFT

## LoRA Settings

- r: 16
- alpha: 32
- dropout: 0.05
- target modules: all-linear
- epochs: 3
- batch size: 1
- gradient accumulation: 4
- learning rate: 2e-4
- max length: 1024

## Interpretation

This run tests whether a larger 42ndMind process-trace dataset improves investigative behavior compared with the v1.8 smoke-test adapter.

The dataset is still narrow and should not be treated as final. It mainly covers supported-domain contradiction, memory, scope, and partial-truth patterns.

## Next Step

Compare:

- base model
- v1.8 smoke-test adapter
- v1.9-safe4 adapter

Then patch 42ndMind so scenarios without generated investigation actions do not crash the batch runner.
"""

report_path.write_text(report, encoding='utf-8')
print('Wrote:', report_path)
files.download(str(report_path))